# Binary Classification with a Tabular Employee Attrition Dataset  

This project involves predicting employee attrition using a binary classification approach. The dataset is synthetically generated from real-world data, making it ideal for exploring various machine learning techniques and feature engineering strategies.  

You can access the dataset [here](https://www.kaggle.com/competitions/playground-series-s3e3/data).  


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split


In [3]:
data = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1677 entries, 0 to 1676
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   id                        1677 non-null   int64 
 1   Age                       1677 non-null   int64 
 2   BusinessTravel            1677 non-null   object
 3   DailyRate                 1677 non-null   int64 
 4   Department                1677 non-null   object
 5   DistanceFromHome          1677 non-null   int64 
 6   Education                 1677 non-null   int64 
 7   EducationField            1677 non-null   object
 8   EmployeeCount             1677 non-null   int64 
 9   EnvironmentSatisfaction   1677 non-null   int64 
 10  Gender                    1677 non-null   object
 11  HourlyRate                1677 non-null   int64 
 12  JobInvolvement            1677 non-null   int64 
 13  JobLevel                  1677 non-null   int64 
 14  JobRole                 

In [5]:
data.shape

(1677, 35)

In [6]:
data.isnull().sum()

id                          0
Age                         0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSinceLastPromotion     0
YearsWithC

In [7]:
X = data.drop(['Attrition', 'id'], axis=1)
y = data['Attrition']

# Veriyi eğitim ve test setlerine böleriz
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# Kategorik değişkenleri dönüştürmek için OrdinalEncoder
ordinal_encoder = OrdinalEncoder()

X_train[X_train.select_dtypes(include=['object']).columns] = \
    ordinal_encoder.fit_transform(X_train.select_dtypes(include=['object']))
X_test[X_test.select_dtypes(include=['object']).columns] = \
    ordinal_encoder.transform(X_test.select_dtypes(include=['object']))


In [9]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Modelleri tanımlayıp eğitiyoruz
rc= RandomForestClassifier(n_estimators=100, random_state=42)
rc.fit(X_train, y_train)

xgb = xgb.XGBClassifier(random_state=42)
xgb.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [10]:
from sklearn.metrics import roc_auc_score

# Tahminler
y_pred1 = rc.predict_proba(X_test)[:, 1]
y_pred2 = xgb.predict_proba(X_test)[:, 1]

# Tahminlerin ortalaması alınır
avg_pred = (y_pred1 + y_pred2 ) / 2

# AUC-ROC skoru hesaplanır
auc_roc_score = roc_auc_score(y_test, avg_pred)
print(f'AUC-ROC score: {auc_roc_score:.3f}')


AUC-ROC score: 0.816


In [13]:
# Test verisindeki kategorik değişkenleri dönüştür
test[test.select_dtypes(include=['object']).columns] = \
    ordinal_encoder.transform(test.select_dtypes(include=['object']))

# Modellerle tahmin
test_pred1 = rc.predict_proba(test.drop(['id'], axis=1))[:, 1]
test_pred2 = xgb.predict_proba(test.drop(['id'], axis=1))[:, 1]
# Tahminlerin ortalaması alınır
test_avg_pred = (test_pred1 + test_pred2 ) / 2


In [15]:
# Submission dosyasını oluştur
submission = pd.DataFrame({'id': test['id'], 'Attrition': test_avg_pred})
submission.to_csv('submission.csv', index=False)


### Conclusion

In this project, we used **RandomForestClassifier** and **XGBoostClassifier** with average ensembling to predict employee attrition.  

- **Validation AUC-ROC:** 0.816  
- **Kaggle AUC-ROC:** 0.88126  

The ensemble approach improved performance, showing good generalization to unseen data.
